In [ ]:
import ast
import numpy as np
import pandas as pd
import torch
from pathlib import Path

from tfmplayground.model import NanoTabPFNModel

DATA_PATH = "data/climate_ttc/climate_2014_2023_final_with_embeddings_lag_3.csv"
EMBED_PREFIX = "embedding_text_lag"
TEXT_LAGS = [f"{EMBED_PREFIX}{lag}" for lag in (1, 2, 3)]
NUMERIC_COLS = ["precip", "humidity", "windspeed"]
TARGET_COL = "temp"

def load_embeddings(path, max_rows=None):
    df = pd.read_csv(path)
    if max_rows is not None:
        df = df.head(max_rows)
    emb_cols = [c for c in df.columns if c.startswith(EMBED_PREFIX)]
    if not emb_cols:
        raise ValueError(f"No embedding cols with prefix {EMBED_PREFIX}")
    for col in emb_cols:
        df[col] = df[col].apply(ast.literal_eval)
    emb = {col: np.stack(df[col].apply(lambda x: np.asarray(x, dtype=np.float32)).to_list()) for col in emb_cols}
    return emb

def load_numeric_and_target(path, max_rows=None):
    df = pd.read_csv(path)
    if max_rows is not None:
        df = df.head(max_rows)
    X_num = df[NUMERIC_COLS].astype(np.float32).to_numpy()
    y = df[TARGET_COL].astype(np.float32).to_numpy()
    return X_num, y

def mean_cosine_sim_matrix(emb_by_col, cols):
    sims = []
    for col in cols:
        X = torch.tensor(emb_by_col[col], dtype=torch.float32)
        X_norm = torch.nn.functional.normalize(X, dim=1)
        sims.append(X_norm @ X_norm.T)
    return torch.stack(sims).mean(dim=0)

def compute_text_attn(emb_by_col, query_idx, key_indices, text_cols):
    sim = mean_cosine_sim_matrix(emb_by_col, text_cols)
    logits = sim[query_idx, key_indices]
    return torch.softmax(logits, dim=-1)  # [len(keys)]

def get_regular_attn(model, X_all, y_train, single_eval_pos):
    device = torch.device("cpu")
    model = model.to(device)
    X_tensor = torch.tensor(X_all, dtype=torch.float32, device=device).unsqueeze(0)
    y_tensor = torch.tensor(y_train, dtype=torch.float32, device=device).unsqueeze(0)
    attn_cache = []
    def hook_fn(_m, _i, output):
        attn_cache.append(output[1].detach())
    hook = model.transformer_encoder.transformer_blocks[-1].self_attention_between_datapoints.register_forward_hook(hook_fn)
    model.eval()
    with torch.no_grad():
        _ = model((X_tensor, y_tensor), single_eval_pos=single_eval_pos, num_mem_chunks=1,
                  attn_weight_external=None, external_gate=None)
    hook.remove()
    if not attn_cache:
        raise RuntimeError("No attention captured")
    attn = sorted(attn_cache, key=lambda t: t.shape[1])[0]  # pick test->train
    return attn.squeeze(0).cpu()  # [q, k]



using GPU backend
Regular attn: [0.08753327 0.09531105 0.0972116  0.10282961 0.08397662 0.08843128
 0.10589065 0.11300346 0.09405861 0.13175383]
Text attn   : [0.06651945 0.08309734 0.09313877 0.10903364 0.10690395 0.10756564
 0.10167246 0.11042695 0.10941805 0.11222371]
Final attn  : [0.07702637 0.08920419 0.09517518 0.10593162 0.09544028 0.09799846
 0.10378155 0.11171521 0.10173833 0.12198877]


In [3]:
# --- Example run ----------------------------------------------------------
max_rows = 128
emb_by_col = load_embeddings(DATA_PATH, max_rows=max_rows)
X_num, y = load_numeric_and_target(DATA_PATH, max_rows=max_rows)
text_cols = [c for c in TEXT_LAGS if c in emb_by_col]
if len(text_cols) < len(TEXT_LAGS):
    missing = set(TEXT_LAGS) - set(text_cols)
    raise ValueError(f"Missing text lag columns: {missing}")

# Use first 10 rows as train, 11th as test
X_train, y_train = X_num[:10], y[:10]
X_test = X_num[10:11]
emb_by_col_11 = {k: v[:11] for k, v in emb_by_col.items()}
single_eval_pos = len(X_train)

# Normalize y like the interface
y_mean, y_std = y_train.mean(), y_train.std(ddof=1) + 1e-8
y_train_norm = (y_train - y_mean) / y_std
X_all = np.concatenate((X_train, X_test))



In [ ]:
# Model
model = NanoTabPFNModel(
    num_attention_heads=6,
    embedding_size=192,
    mlp_hidden_size=768,
    num_layers=6,
    num_outputs=100,
)

device = torch.device("cpu")
model = model.to(device)
X_tensor = torch.tensor(X_all, dtype=torch.float32, device=device).unsqueeze(0)
y_tensor = torch.tensor(y_train, dtype=torch.float32, device=device).unsqueeze(0)
attn_cache = []
def hook_fn(_m, _i, output):
    attn_cache.append(output[1].detach())

# attach hook to the last layer's cross-datapoint attention
hook = model.transformer_encoder.transformer_blocks[-1].self_attention_between_datapoints.register_forward_hook(hook_fn)
model.eval()
with torch.no_grad():
    _ = model((X_tensor, y_tensor), single_eval_pos=single_eval_pos, num_mem_chunks=1,
                attn_weight_external=None, external_gate=None)
hook.remove()
if not attn_cache:
    raise RuntimeError("No attention captured")
attn = sorted(attn_cache, key=lambda t: t.shape[1])[0]  # pick test->train

In [15]:
attn_cache[0].shape

torch.Size([4, 10, 10])

In [11]:
attn.shape

torch.Size([4, 1, 10])

In [ ]:
# Model
model = NanoTabPFNModel(
    num_attention_heads=6,
    embedding_size=192,
    mlp_hidden_size=768,
    num_layers=6,
    num_outputs=100,
)

# Regular attention (row 11 -> rows 1-10)
attn_regular = get_regular_attn(model, X_all, y_train_norm, single_eval_pos)[0]  # [10]

# Text attention (row 11 -> rows 1-10)
attn_text = compute_text_attn(
    emb_by_col_11,
    query_idx=10,
    key_indices=list(range(10)),
    text_cols=text_cols,
)

# Blend (example 50/50)
attn_final = 0.5 * attn_regular.numpy().reshape(-1) + 0.5 * attn_text.numpy().reshape(-1)

print("Regular attn:", attn_regular.numpy().reshape(-1))
print("Text attn   :", attn_text.numpy().reshape(-1))
print("Final attn  :", attn_final)
